# openEO backup — post-fire recovery time series

This notebook provides the live CDSE replacement for the core time-series idea in **Practical 04**.

It creates the same late-summer seasonal windows for **2019–2025**, calculates median Sentinel-2 composites, derives NBR/NDVI/NDMI, and aggregates the indices over EFFIS target polygon **240575**.

The canonical Practical 04 contains the fuller impact-zone and reference-zone analysis. This backup deliberately keeps the workflow smaller.

### Kernel

Run this fallback in the **same Python 3 kernel used for the canonical GEE notebooks**.

The cell below installs only the lightweight `openeo` Python client when it is missing. Do not install GeoPandas, Rasterio, xarray or other compiled geospatial packages into CDSE's dedicated OpenEO kernel just for this course.

In [ ]:
import importlib.util
import subprocess
import sys

# The openEO client is intentionally the only package installed at runtime.
# The canonical Python 3 course kernel already provides the geospatial stack.
if importlib.util.find_spec("openeo") is None:
    print("Installing the openEO Python client in the current session...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openeo>=0.50,<1",
    ])

required_existing = ["geopandas", "numpy", "matplotlib"]
missing_existing = [
    name for name in required_existing
    if importlib.util.find_spec(name) is None
]

if missing_existing:
    raise RuntimeError(
        "This fallback should run in the same Python 3 kernel as the GEE "
        "practicals. Missing core package(s): "
        + ", ".join(missing_existing)
        + ". Do not repair this by installing compiled geospatial packages "
        "into the dedicated OpenEO kernel; switch to the course Python 3 kernel."
    )

print("Fallback kernel ready.")

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import openeo
import pandas as pd

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd(),
        Path.home() / "mystorage" / "fire-school",
    ]
    for candidate in candidates:
        if (candidate / "data/effis/Galicica.gpkg").exists():
            return candidate
    raise FileNotFoundError("Could not find the fire-school repository.")

REPO_ROOT = find_repo_root()
effis = gpd.read_file(REPO_ROOT / "data/effis/Galicica.gpkg").to_crs("EPSG:4326")

target = effis[effis["id"].astype(str) == "240575"].copy()
if target.empty:
    raise RuntimeError("EFFIS target polygon 240575 was not found.")

target_geom = target.geometry.iloc[0]
west, south, east, north = target.total_bounds

BBOX = {
    "west": float(west),
    "south": float(south),
    "east": float(east),
    "north": float(north),
    "crs": "EPSG:4326",
}

print("Target polygon loaded.")

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("openEO authenticated.")

## Seasonal intervals

The interval end is exclusive. Using 16 October therefore includes observations through 15 October, matching the canonical practical.

In [ ]:
YEARS = list(range(2019, 2026))

intervals = [
    [f"{year}-08-20", f"{year}-10-16"]
    for year in YEARS
]

labels = [f"{year}-09-15" for year in YEARS]

intervals

## Load, mask and aggregate Sentinel-2

In [ ]:
MAX_CLOUD = 80

scl = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=BBOX,
    temporal_extent=[intervals[0][0], intervals[-1][1]],
    bands=["SCL"],
    max_cloud_cover=MAX_CLOUD,
)

cloud_mask = scl.process(
    "to_scl_dilation_mask",
    data=scl,
    kernel1_size=17,
    kernel2_size=77,
    mask1_values=[2, 4, 5, 6, 7],
    mask2_values=[3, 8, 9, 10, 11],
    erosion_kernel_size=3,
)

s2 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=BBOX,
    temporal_extent=[intervals[0][0], intervals[-1][1]],
    bands=["B04", "B08", "B11", "B12"],
    max_cloud_cover=MAX_CLOUD,
)

seasonal = (
    s2
    .mask(cloud_mask)
    .aggregate_temporal(
        intervals=intervals,
        reducer="median",
        labels=labels,
    )
)

print("Seasonal process graph ready.")

## Calculate NBR, NDVI and NDMI

In [ ]:
def normalized_difference(cube, band_a, band_b):
    a = cube.band(band_a)
    b = cube.band(band_b)
    return (a - b) / (a + b)

nbr = normalized_difference(seasonal, "B08", "B12").add_dimension(
    name="bands", label="NBR", type="bands"
)
ndvi = normalized_difference(seasonal, "B08", "B04").add_dimension(
    name="bands", label="NDVI", type="bands"
)
ndmi = normalized_difference(seasonal, "B08", "B11").add_dimension(
    name="bands", label="NDMI", type="bands"
)

indices = nbr.merge_cubes(ndvi).merge_cubes(ndmi)

timeseries = indices.aggregate_spatial(
    geometries=target_geom,
    reducer="mean",
)

print("Zonal time-series graph ready.")

## Execute as a CSV batch job

The first live request can take several minutes. The result is cached under persistent `~/mystorage/geo_adapt_openeo_cache/` so rerunning the notebook does not repeat the cloud processing.

In [ ]:
mystorage = Path.home() / "mystorage"
CACHE_DIR = (
    mystorage / "geo_adapt_openeo_cache"
    if mystorage.exists()
    else Path("/tmp/geo_adapt_openeo")
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = CACHE_DIR / "recovery_timeseries.csv"

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    print("Using cached openEO result:", OUTPUT)
else:
    print(
        "No cached result found. The first live recovery request can take "
        "several minutes; later runs reuse this CSV."
    )
    job = timeseries.execute_batch(
        out_format="CSV",
        title="GEO-ADAPT openEO backup: recovery time series",
    )
    downloaded = job.get_results().download_file(str(OUTPUT))
    print("Downloaded:", downloaded)

## Inspect the returned table

CDSE/openEO CSV metadata can evolve, so first inspect the returned columns rather than assuming a fixed export layout.

In [ ]:
df = pd.read_csv(OUTPUT)
display(df.head(10))
print("Columns:", list(df.columns))

In [ ]:
date_candidates = [
    c for c in df.columns
    if c.lower() in {"date", "time", "datetime"} or "date" in c.lower()
]

if date_candidates:
    date_col = date_candidates[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = (
        df
        .dropna(subset=[date_col])
        .sort_values(date_col)
        .reset_index(drop=True)
    )

    value_cols = [
        c for c in df.columns
        if c.upper() in {"NBR", "NDVI", "NDMI"}
    ]

    if value_cols:
        display(df[[date_col] + value_cols])

        for column in value_cols:
            values = pd.to_numeric(df[column], errors="coerce")
            valid = values.between(-1, 1) | values.isna()
            excluded = int((~valid).sum())
            values = values.where(valid)

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(df[date_col], values, marker="o")
            ax.set_title(f"Target polygon — {column}")
            ax.set_ylabel(column)
            ax.set_xlabel("Season")
            ax.set_ylim(-1, 1)
            ax.grid(alpha=0.25)
            plt.show()

            if excluded:
                print(
                    f"{column}: excluded {excluded} value(s) outside "
                    "the physical index range [-1, 1]."
                )
    else:
        print("Index columns were not named NBR/NDVI/NDMI in this export.")
        print("Use the table above to identify the three value columns.")
else:
    print("No obvious date column was found automatically.")
    print("Use the table above; the backend export layout may have changed.")

## Interpretation

The expected qualitative pattern is:

- relatively stable pre-fire seasonal values;
- a clear 2024 disturbance;
- movement back toward the pre-fire range in 2025.

Do not convert that into a claim of complete ecological recovery. This backup measures spectral response over the EFFIS target polygon.